# Klipper Bed Clear - training

Train a tiny CNN to classify the bed as `clear` or `not_clear`.

## Data

Upload `frames.zip` with this layout:

```
clear/*.jpg
not_clear/*.jpg
```

The original `BedClearframes/` and `NotClearFrames/` folder names also work.

**Run the audit first.** Sequential video frames are often near-duplicates. If the audit reports only a few unique images, stop and collect more scenes; training on duplicates gives meaningless accuracy.

In [ ]:
%cd /content
!nvidia-smi || true


In [ ]:
%cd /content
!rm -rf /content/KlipperBedCheck
!git clone https://github.com/weegeeday/KlipperBedCheck.git /content/KlipperBedCheck
%cd /content/KlipperBedCheck
!pip -q install -r /content/KlipperBedCheck/training/requirements.txt


## Upload frames

Zip the `clear` and `not_clear` folders and upload the zip here.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pathlib, shutil, zipfile
from collections import Counter

zip_name = next(iter(uploaded))
shutil.rmtree('/content/frames', ignore_errors=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content/frames')

names = {'clear', 'bedclearframes', 'bed_clear',
         'not_clear', 'notclearframes', 'notclear', 'not clear'}
candidates = [p for p in pathlib.Path('/content/frames').rglob('*')
              if p.is_dir() and p.name.lower() in names]
if not candidates:
    raise SystemExit('No clear/not_clear folders found in the zip')
data_dir = Counter(p.parent for p in candidates).most_common(1)[0][0]
print('data_dir =', data_dir)

In [ ]:
!python training/audit_dataset.py --data {data_dir} --out training/reports

### Stop here if the unique counts are low

A 200-frame video can contain only two unique scenes. Training on that will not tell you anything useful about real accuracy.

Also check `training/reports/contact_sheet.jpg` to confirm the camera framing and ROI.

## Train

Set `--roi` to the part of the frame that contains the bed. `0,0,1,1` means the full frame. Deployment must use the same ROI.

In [ ]:
!python training/train.py --data {data_dir} --out training/runs/run1 --epochs 40 --img-size 128 --batch-size 32 --roi 0,0,1,1


In [ ]:
import json
from IPython.display import Image, display

metrics = json.load(open('training/runs/run1/metrics.json'))
print(json.dumps(metrics, indent=2))
display(Image('training/runs/run1/confusion_matrix.png'))
display(Image('training/runs/run1/samples.png'))

In [ ]:
!cd /content/KlipperBedCheck && zip -qr /content/bedclear_run1.zip training/runs/run1
from google.colab import files
files.download('/content/bedclear_run1.zip')

## Next

- `not_clear_recall` is the metric that matters most; a false clear is the dangerous error.
- Keep the image-diff result as a fallback in the deployed checker.
- Collect more sessions and lighting states before trusting the test accuracy.